In [16]:
import jax.numpy as jnp

## 🔍 Understanding `jnp.sum` with `axis` in JAX

This notebook demonstrates how summing over different axes affects the result, using softmax log-likelihood and a general matrix.

---

### ✅ Example 1: Log-likelihood of correct classes

We simulate probabilities from a softmax layer and use one-hot labels to extract the log-probability of the correct class for each sample.


In [18]:
# Suppose 3 data points and 3 classes
p_all = jnp.array([
    [0.7, 0.2, 0.1],
    [0.1, 0.8, 0.1],
    [0.2, 0.3, 0.5]
])

y_onehot = jnp.array([
    [1, 0, 0],   # true class is 0
    [0, 1, 0],   # true class is 1
    [0, 0, 1]    # true class is 2
])

log_p_all = jnp.log(p_all)
print("jnp.log(p_all) =\n", log_p_all)

print("")

y_times_log_p = y_onehot * log_p_all
print("y_onehot * jnp.log(p_all) =\n", y_times_log_p)


jnp.log(p_all) =
 [[-0.35667497 -1.609438   -2.3025851 ]
 [-2.3025851  -0.22314353 -2.3025851 ]
 [-1.609438   -1.2039728  -0.6931472 ]]

y_onehot * jnp.log(p_all) =
 [[-0.35667497 -0.         -0.        ]
 [-0.         -0.22314353 -0.        ]
 [-0.         -0.         -0.6931472 ]]


In [19]:
print("Sum over axis=0:\n", jnp.sum(y_times_log_p, axis=0))  # sum across rows (per class)
print("")
print("Sum over axis=1:\n", jnp.sum(y_times_log_p, axis=1))  # sum across columns (per sample)
print("")
print("Sum over axis=-1:\n", jnp.sum(y_times_log_p, axis=-1))  # same as axis=1
print("")
print("Total sum:\n", jnp.sum(y_times_log_p))  # scalar total


Sum over axis=0:
 [-0.35667497 -0.22314353 -0.6931472 ]

Sum over axis=1:
 [-0.35667497 -0.22314353 -0.6931472 ]

Sum over axis=-1:
 [-0.35667497 -0.22314353 -0.6931472 ]

Total sum:
 -1.2729657


#### 💡 Explanation

Since each row in `y_onehot` contains exactly one 1, and the 1s are in different columns, we are only ever picking one log-probability per row.

This results in a diagonal matrix in `y_times_log_p`, e.g.:

```python
[[-0.357,  0,      0     ],
 [ 0,     -0.223,  0     ],
 [ 0,      0,     -0.693]]
```

#### ➡️ In this case, the final scalar sum is the same, no matter whether you use axis=0, axis=1, axis=-1, or no axis at all:

```
Sum over axis=0: [-0.357, -0.223, -0.693]
Sum over axis=1: [-0.357, -0.223, -0.693]
Total sum:       -1.273
```

#### 📌 When your matrix is diagonal like this (1 non-zero per row/column), it doesn't matter how you sum — the result is the same values just arranged differently.
_____________________

### ✅ Example 2: General Matrix to Show Axis Meaning

In [ ]:
arr = jnp.array([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
])

#### 🖊️ axis=0: sum down the columns

```python
jnp.sum(arr, axis=0)  # [1+4+7, 2+5+8, 3+6+9] → [12, 15, 18]
```

#### 🖊️ axis=1: sum across the rows

```python
jnp.sum(arr, axis=1)  # [1+2+3, 4+5+6, 7+8+9] → [6, 15, 24]
```

#### 🖊️ axis=-1: same as axis=1 (last axis)

```python
jnp.sum(arr, axis=-1)  # → [6, 15, 24]
```
#### 🖊️ No axis specified: sum all elements

```python
jnp.sum(arr)  # → 45
``` 

In [21]:
print("Original array:\n", arr)

print("\nSum over axis=0 (down columns):\n", jnp.sum(arr, axis=0))
print("Explanation: sums down each column → [1+4+7, 2+5+8, 3+6+9]")

print("\nSum over axis=1 (across rows):\n", jnp.sum(arr, axis=1))
print("Explanation: sums across each row → [1+2+3, 4+5+6, 7+8+9]")

print("\nSum over axis=-1 (same as axis=1 here):\n", jnp.sum(arr, axis=-1))

print("\nTotal sum:\n", jnp.sum(arr))

Original array:
 [[1 2 3]
 [4 5 6]
 [7 8 9]]

Sum over axis=0 (down columns):
 [12 15 18]
Explanation: sums down each column → [1+4+7, 2+5+8, 3+6+9]

Sum over axis=1 (across rows):
 [ 6 15 24]
Explanation: sums across each row → [1+2+3, 4+5+6, 7+8+9]

Sum over axis=-1 (same as axis=1 here):
 [ 6 15 24]

Total sum:
 45


________________________
| Axis      | What happens                          | Use-case                                     |
| --------- | ------------------------------------- | -------------------------------------------- |
| `axis=0`  | Sum **down columns** (combine rows)   | Sum over data points (e.g. per-class stats)  |
| `axis=1`  | Sum **across rows** (combine columns) | Sum over features/classes per data point     |
| `axis=-1` | Sum over **last axis**                | Use for generality when axis position varies |
| No axis   | Sum **all elements**                  | Total scalar result                          |


✅ Use `axis=1` when you want per-sample values in classification (e.g., softmax across classes).

✅ Use `axis=0` when you want per-class summaries over many samples.

#### 🧪 Tip:

Always print shapes to confirm what you're working with:

```python
print("Shape of logits:", logits.shape)
print("Shape of log-likelihood per sample:", loglik.shape)
print("Shape of y_onehot:", y_onehot.shape)

### Log prior 

In [31]:
import jax.numpy as jnp

alpha = 1
w_flat = jnp.array([0., 0., 0., 0., 0., 0., 0., 0.])

log_npdf = lambda x, m, v: -0.5 * (x - m)**2 / v - 0.5 * jnp.log(2 * jnp.pi * v)

log_prior = log_npdf(w_flat, 0, 1 / alpha)
print("log_prior per weight =", log_prior)
print("total log_prior =", jnp.sum(log_prior, axis=-1))


log_prior per weight = [-0.9189385 -0.9189385 -0.9189385 -0.9189385 -0.9189385 -0.9189385
 -0.9189385 -0.9189385]
total log_prior = -7.351508


## 🔍 Understanding `jnp.sum(..., axis=...)` in JAX

You're now exploring how `jnp.sum(..., axis=...)` behaves — which is only meaningful if the input is more than 1D.

---

### 🧠 Key Point

If your array is 1D:

```python
log_prior = log_npdf(w_flat, 0, 1 / alpha)
```

Then:

- `w_flat.shape → (8,)` → a 1D array
- `log_prior.shape → (8,)` → log-probability for each weight

So if you try:

```python
jnp.sum(log_prior, axis=1)
```

💥 You'll get:

```text
AxisError: axis 1 is out of bounds for array of dimension 1
```

Because axis `1` doesn't exist in a 1D array.

---

### ✅ What you can do

You can sum all elements:

```python
jnp.sum(log_prior)              # ✅ total scalar sum
jnp.sum(log_prior, axis=0)      # ✅ same result (for 1D)
```

---

## 🧪 Example: Demonstrating axis=0, axis=1, and axis=-1

You need a **2D array** to see meaningful axis behavior:

```python
log_prior_matrix = jnp.stack([
    log_npdf(jnp.array([0., 1., 2.]), 0, 1),  # row 0
    log_npdf(jnp.array([3., 4., 5.]), 0, 1)   # row 1
])  # shape (2, 3)
```

```python
print("log_prior_matrix:\n", log_prior_matrix)

print("\nSum over axis=0:", jnp.sum(log_prior_matrix, axis=0))  # column-wise
print("Sum over axis=1:", jnp.sum(log_prior_matrix, axis=1))    # row-wise
print("Sum over axis=-1:", jnp.sum(log_prior_matrix, axis=-1))  # same as axis=1
print("Total sum:", jnp.sum(log_prior_matrix))
```

---

### 📌 Output (conceptual)

```text
log_prior_matrix =
[[-0.92, -1.42, -3.42],
 [-5.42, -8.42, -11.42]]

Sum over axis=0 = [-6.34, -9.84, -14.84]   # down columns
Sum over axis=1 = [-5.76, -25.26]          # across rows
Sum over axis=-1 = [-5.76, -25.26]         # same as axis=1
Total sum = -31.02                         # sum of all elements
```

---

### ✅ Summary: How to Think About `axis`

| Axis        | What it does                | Meaning in practice                            |
|-------------|-----------------------------|-------------------------------------------------|
| `axis=0`    | Sum **down columns**        | Sum across samples for each feature/class       |
| `axis=1`    | Sum **across rows**         | Sum over features for each sample               |
| `axis=-1`   | Same as `axis=1` (2D case)  | Always targets the last axis                    |
| `axis=None` | Sum **all elements**        | Returns a total scalar                         |

---

✅ Use axis control to choose **how you aggregate** over a matrix — whether row-wise, column-wise, or all together.


## ✅ When Likelihood Is 2D: Understanding `axis` in Practice

There are practical situations where the **log-likelihood becomes a 2D array**, and using `axis` correctly is essential.

---

### 🧠 Scenario: Mini-batch or Monte Carlo Log-Likelihoods

Suppose you're computing the **log-likelihood of multiple models (or samples) on multiple data points**. This happens often in:

- Monte Carlo sampling from the posterior
- Ensembles
- Variational inference
- Bayesian model averaging

Let:

- `S` = number of weight samples or models
- `N` = number of data points

Then:

```python
log_likelihoods.shape = (S, N)  # sample × datapoint
```

---

### 🧪 Example: 3 Samples × 4 Data Points

```python
import jax.numpy as jnp

# log-likelihoods for 3 models/samples across 4 data points
log_likelihoods = jnp.array([
    [-1.0, -0.5, -2.0, -0.1],   # sample 0
    [-1.2, -0.4, -2.1, -0.2],   # sample 1
    [-0.9, -0.6, -1.9, -0.3],   # sample 2
])  # shape (3, 4)
```

---

### 🔎 Use of Different `axis` Values

#### ✅ Sum over `axis=1`: total per model (across data points)
```python
jnp.sum(log_likelihoods, axis=1)
```
Output:
```text
[-3.6, -3.9, -3.7]  # total log-likelihood for each sample
```

#### ✅ Sum over `axis=0`: total per data point (across models)
```python
jnp.sum(log_likelihoods, axis=0)
```
Output:
```text
[-3.1, -1.5, -6.0, -0.6]  # total log-likelihood across samples per point
```

#### ✅ No axis: total scalar log-likelihood
```python
jnp.sum(log_likelihoods)
```
Output:
```text
-11.2
```

---

### ✅ Summary Table

| Expression                          | What it gives                          |
|-------------------------------------|----------------------------------------|
| `jnp.sum(log_likelihoods, axis=1)` | total per **model/sample**             |
| `jnp.sum(log_likelihoods, axis=0)` | total per **data point**               |
| `jnp.sum(log_likelihoods)`         | **total scalar** over all              |

---

### 🧠 When Does This Arise?

| Context                     | What the axes represent                |
|----------------------------|----------------------------------------|
| Monte Carlo estimation     | Each row is a sample from posterior    |
| Ensembles                  | Each row is a model                    |
| Batches in training        | Each row is a batch or forward pass    |
| Variational inference      | Samples from q(w), rows are particles  |

---

✅ When working with arrays of log-likelihoods (or losses), always **inspect shapes first**, then choose the appropriate axis based on **how you want to aggregate**.


## ✅ When to Use `axis` in Likelihood Computations (Gaussian Example)

This example shows when and how to use `axis=0`, `axis=1`, or `axis=-1` when computing log-likelihoods, using a **Gaussian likelihood** with **multiple model samples** (e.g. Monte Carlo or ensemble setting).

---

### 🧠 Problem Setup

We assume the data is generated by a Gaussian likelihood:

$$
y_n \sim \mathcal{N}(\mu_n, \sigma^2)
$$

The log-likelihood for a single data point is:

$$
\log p(y_n \mid \mu_n, \sigma^2) = -\frac{1}{2} \log(2\pi \sigma^2) - \frac{1}{2\sigma^2} (y_n - \mu_n)^2
$$

We’ll evaluate this for multiple model predictions (`mu_samples`) and data points (`y_true`).

---

### 🧪 JAX Code Example

```python
import jax.numpy as jnp

# True outputs (y_n) and predictions (mu_n) from 3 sampled models
y_true = jnp.array([1.0, 2.0, 3.0, 4.0])  # shape (4,)
mu_samples = jnp.array([
    [1.1, 1.9, 2.8, 3.9],   # sample 0
    [1.2, 2.1, 3.2, 4.1],   # sample 1
    [0.9, 2.2, 3.1, 4.0],   # sample 2
])  # shape (3, 4)

sigma2 = 0.1

log_npdf = lambda y, mu, sigma2: -0.5 * jnp.log(2 * jnp.pi * sigma2) - 0.5 * ((y - mu)**2) / sigma2

# Broadcasting y_true to shape (3, 4) for pairwise comparisons
log_likelihoods = log_npdf(y_true[None, :], mu_samples, sigma2)  # shape (3, 4)
```

---

### 🔎 What is `log_likelihoods.shape`?

- Shape: `(3, 4)`
- Rows = model samples (`S = 3`)
- Columns = data points (`N = 4`)

---

### ✅ Using `axis`

| Expression                                 | Meaning                                | Shape        |
|--------------------------------------------|----------------------------------------|--------------|
| `jnp.sum(log_likelihoods, axis=1)`         | Sum over columns → total per sample    | shape = (3,) |
| `jnp.sum(log_likelihoods, axis=0)`         | Sum over rows → total per data point   | shape = (4,) |
| `jnp.sum(log_likelihoods)`                 | Total over all samples and points      | scalar       |

---

### 📌 Output Example

```python
print("Log-likelihoods:\n", log_likelihoods)

print("\nSum over axis=1 (per sample):", jnp.sum(log_likelihoods, axis=1))  # shape (3,)
print("Sum over axis=0 (per data point):", jnp.sum(log_likelihoods, axis=0))  # shape (4,)
print("Total log-likelihood:", jnp.sum(log_likelihoods))  # scalar
```

---

### 📓 Summary in Math

Let:

- $$ \ell_{s,n} = \log p(y_n \mid \mu_{s,n}) $$

Then:

- **Per-sample log-likelihood**:  
  $$
  \text{LL}_s = \sum_{n=1}^N \ell_{s,n} \quad \Rightarrow \texttt{axis=1}
  $$

- **Per-data-point log-likelihood**:
  $$
  \text{LL}_n = \sum_{s=1}^S \ell_{s,n} \quad \Rightarrow \texttt{axis=0}
  $$

- **Total log-likelihood**:
  $$
  \text{LL}_{\text{total}} = \sum_{s=1}^S \sum_{n=1}^N \ell_{s,n} \quad \Rightarrow \texttt{no axis}
  $$

---

### ✅ Practical Usage

```python
# Monte Carlo estimate of log p(y | x)
jnp.mean(jnp.sum(log_likelihoods, axis=1))  # Average over samples
```

---

### ✅ Axis Summary Table

| Axis        | What it does                | Meaning in practice                            |
|-------------|-----------------------------|-------------------------------------------------|
| `axis=0`    | Sum **down columns**        | Sum across samples for each data point         |
| `axis=1`    | Sum **across rows**         | Sum over all data points for each sample       |
| `axis=-1`   | Same as `axis=1` here       | Always targets the last axis                   |
| `axis=None` | Sum **all elements**        | Returns scalar total log-likelihood            |

---

✅ Tip: Always inspect shapes using `.shape` and `print(...)` to know which `axis` you need!
